# Geometric Morphometric Data Augmentation & Imbalance Classification
## Experiment 1: Original Data (Baseline - Class Weighted)

### 📌 Study Overview & Experimental Objective
* **Dataset Size & Class Distribution**: Full Imbalanced Dataset (ONC: 5,393 [85.1%], MAM: 722 [11.4%], SAM: 220 [3.5%])
* **Feature Selection**: Geometric Morphometric Procrustes Coordinates excluding noisy/collinear landmarks (**LM11**, **LM21**, **LM22**) $\rightarrow$ **41 total features** (`GPA_X1`..`GPA_Y20`, `Centroid_Size`, `age_month`, `gender_num`).
* **Primary Metric Focus**: **SAM Recall**, **SAM Precision**, **SAM F1**, **Macro F1**, and **Balanced Accuracy** (avoiding raw overall accuracy which masks minor class performance).

---
### 🔄 Experimental Pipeline
```text
Landmarks (GPA) -> Exclude Noise (LM11,21,22) -> 5-Fold CV -> Class-Weighted XGBoost -> Evaluation
```


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score, balanced_accuracy_score, confusion_matrix
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import BorderlineSMOTE, SMOTE
from imblearn.combine import SMOTETomek
import warnings
warnings.filterwarnings("ignore")

# Global Reproducibility Seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("✅ Libraries loaded successfully.")


✅ Libraries loaded successfully.

In [2]:
# Load Cleaned Geometric Morphometric Data
data_path = r"e:\ap-govt-EDA-model-tainning-testing\GM_out_of_sample_classification-main\GM_out_of_sample_classification-ap-govt\outliers_quality_of_data_extraction\final_cleaned_dataset_full.csv"
df = pd.read_csv(data_path)

# Encode Gender
df['gender_num'] = df['gender'].map({'M': 0, 'F': 1, 'Male': 0, 'Female': 1}).fillna(0)

# Exclude Noisy & Collinear Landmarks (LM11, LM21, LM22)
exclude_lms = ['11', '21', '22']
exclude_cols = [f'GPA_X{lm}' for lm in exclude_lms] + [f'GPA_Y{lm}' for lm in exclude_lms]

feature_cols = [c for c in df.columns if c.startswith('GPA_') and c not in exclude_cols] + ['Centroid_Size', 'age_month', 'gender_num']
target_col = 'Class_Type'

print(f"📊 Cleaned Dataset Shape: {df.shape}")
print(f"🧬 Selected Feature Count: {len(feature_cols)} (Excluding LM11, LM21, LM22)")
print("🏷️ Full Dataset Class Distribution:")
print(df[target_col].value_counts())


📊 Cleaned Dataset Shape: (6207, 104)
🧬 Selected Feature Count: 41 (Excluding LM11, LM21, LM22)
🏷️ Full Dataset Class Distribution:
onc    5393
mam     722
sam     220
Name: Class_Type, dtype: int64

In [3]:
# Experiment 1 Setup: Full Imbalanced Dataset with Class-Weighted XGBoost
X_exp = df[feature_cols].values
y_exp = df['Class_Type'].map({'onc': 0, 'mam': 1, 'sam': 2}).values

print(f"Experiment 1 Input Matrix: {X_exp.shape}")
print("No oversampling applied; class balancing handled via compute_sample_weight('balanced').")


Experiment 1 Input Matrix: (6207, 41)
No oversampling applied; class balancing handled via compute_sample_weight('balanced').

In [4]:
# Stratified 5-Fold Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

recalls, precisions, f1s_sam, macro_f1s, balanced_accs = [], [], [], [], []
cm_total = np.zeros((3, 3), dtype=int)
class_names = ['ONC', 'MAM', 'SAM']

print(f"🚀 Running 5-Fold Stratified Cross-Validation (Exp 1: Original Data (Baseline - Class Weighted))...\n")

for fold, (train_idx, test_idx) in enumerate(cv.split(X_exp, y_exp), 1):
    X_tr, y_tr = X_exp[train_idx], y_exp[train_idx]
    X_te, y_te = X_exp[test_idx], y_exp[test_idx]
    
    # Apply Resampling / Weighting strictly INSIDE training fold to prevent data leakage
    if "exp1" == "exp1":
        sample_weights = compute_sample_weight('balanced', y_tr)
        X_res, y_res = X_tr, y_tr
    elif "exp1" == "exp2":
        X_res, y_res = X_tr, y_tr
    elif "exp1" == "exp3":
        bsmote = BorderlineSMOTE(sampling_strategy={2: 400}, random_state=RANDOM_STATE, k_neighbors=3)
        X_res, y_res = bsmote.fit_resample(X_tr, y_tr)
    elif "exp1" == "exp4":
        bsmote = BorderlineSMOTE(sampling_strategy={2: 577}, random_state=RANDOM_STATE, k_neighbors=3)
        X_res, y_res = bsmote.fit_resample(X_tr, y_tr)
    elif "exp1" == "exp5":
        smote_tomek = SMOTETomek(random_state=RANDOM_STATE)
        X_res, y_res = smote_tomek.fit_resample(X_tr, y_tr)
    elif "exp1" == "exp6":
        bsmote = BorderlineSMOTE(random_state=RANDOM_STATE, k_neighbors=3)
        X_res, y_res = bsmote.fit_resample(X_tr, y_tr)

    # Instantiate XGBoost Model
    model = XGBClassifier(
        objective='multi:softprob',
        eval_metric='mlogloss',
        random_state=RANDOM_STATE,
        n_estimators=100,
        learning_rate=0.1,
        max_depth=4
    )
    
    if "exp1" == "exp1":
        model.fit(X_tr, y_tr, sample_weight=sample_weights)
    else:
        model.fit(X_res, y_res)
    
    # Evaluate on unseen test fold
    y_pred = model.predict(X_te)
    
    rec = recall_score(y_te, y_pred, labels=[2], average=None)[0]
    prec = precision_score(y_te, y_pred, labels=[2], average=None, zero_division=0)[0]
    f1_s = f1_score(y_te, y_pred, labels=[2], average=None)[0]
    m_f1 = f1_score(y_te, y_pred, average='macro')
    b_acc = balanced_accuracy_score(y_te, y_pred)
    
    recalls.append(rec)
    precisions.append(prec)
    f1s_sam.append(f1_s)
    macro_f1s.append(m_f1)
    balanced_accs.append(b_acc)
    cm_total += confusion_matrix(y_te, y_pred, labels=[0, 1, 2])
    
    print(f"Fold {fold}: SAM Recall={rec:.4f} | SAM Prec={prec:.4f} | SAM F1={f1_s:.4f} | Macro F1={m_f1:.4f} | Bal Acc={b_acc:.4f}")


🚀 Running 5-Fold Stratified Cross-Validation (Exp 1: Original Data (Baseline - Class Weighted))...

Fold 1: SAM Recall=0.4091 | SAM Prec=0.2903 | SAM F1=0.3396 | Macro F1=0.5724 | Bal Acc=0.6348
Fold 2: SAM Recall=0.4545 | SAM Prec=0.2899 | SAM F1=0.3540 | Macro F1=0.5764 | Bal Acc=0.6534
Fold 3: SAM Recall=0.2093 | SAM Prec=0.1915 | SAM F1=0.2000 | Macro F1=0.5180 | Bal Acc=0.5596
Fold 4: SAM Recall=0.3488 | SAM Prec=0.2727 | SAM F1=0.3061 | Macro F1=0.5593 | Bal Acc=0.6204
Fold 5: SAM Recall=0.3953 | SAM Prec=0.3469 | SAM F1=0.3696 | Macro F1=0.5819 | Bal Acc=0.6273

In [5]:
# Compute Aggregated Metrics Across Folds
mean_sam_rec = np.mean(recalls)
mean_sam_prec = np.mean(precisions)
mean_sam_f1 = np.mean(f1s_sam)
mean_macro_f1 = np.mean(macro_f1s)
mean_bal_acc = np.mean(balanced_accs)

print("\n" + "="*65)
print(f"🏆 SUMMARY RESULTS: EXP 1 - ORIGINAL DATA (BASELINE - CLASS WEIGHTED)")
print("="*65)
print(f"🎯 SAM Recall        : {mean_sam_rec:.4f} ({mean_sam_rec*100:.2f}%)")
print(f"🎯 SAM Precision     : {mean_sam_prec:.4f} ({mean_sam_prec*100:.2f}%)")
print(f"🎯 SAM F1-Score      : {mean_sam_f1:.4f} ({mean_sam_f1*100:.2f}%)")
print(f"📈 Macro F1-Score    : {mean_macro_f1:.4f} ({mean_macro_f1*100:.2f}%)")
print(f"⚖️  Balanced Accuracy: {mean_bal_acc:.4f} ({mean_bal_acc*100:.2f}%)")
print("="*65)

# Plot Aggregated Confusion Matrix
plt.figure(figsize=(7, 5))
sns.heatmap(cm_total, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f"Confusion Matrix (5-Fold Total) - Exp 1: Original Data (Baseline - Class Weighted)", fontsize=12, fontweight='bold')
plt.xlabel("Predicted Class", fontsize=11)
plt.ylabel("True Class", fontsize=11)
plt.tight_layout()
plt.show()


🏆 SUMMARY RESULTS: EXP 1 - ORIGINAL DATA (BASELINE - CLASS WEIGHTED)
🎯 SAM Recall        : 0.3634 (36.34%)
🎯 SAM Precision     : 0.2783 (27.83%)
🎯 SAM F1-Score      : 0.3139 (31.39%)
📈 Macro F1-Score    : 0.5616 (56.16%)
⚖️  Balanced Accuracy: 0.6191 (61.91%)